In [1]:
# Cella 1: Setup
import sys
from pathlib import Path
import torch
from transformers import TrainingArguments, Trainer

ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
from teacher_finetune_headtail import (
    load_flat_dataset, TeacherModelConfig, build_teacher_model,
    build_teacher_tokenizer, build_collator, compute_metrics,
    get_llrd_optimizer_parameters, bf16_supported
)

paths = get_paths(ROOT)

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_ds = load_flat_dataset(paths.data_processed / "train_100k_balanced.parquet")
val_ds = load_flat_dataset(paths.data_processed / "val_real.parquet")

# Check rapido
print(f"Train size: {len(train_ds)} (Balanced)")
print(f"Val size: {len(val_ds)} (Imbalanced Real)")

Generating train split: 100000 examples [00:00, 205945.37 examples/s]
Map: 100%|██████████| 100000/100000 [00:03<00:00, 28335.13 examples/s]
Generating train split: 5000 examples [00:00, 199972.54 examples/s]
Map: 100%|██████████| 5000/5000 [00:00<00:00, 33109.08 examples/s]

Train size: 100000 (Balanced)
Val size: 5000 (Imbalanced Real)


In [3]:
MODEL_NAME = "bert-base-uncased"
tokenizer = build_teacher_tokenizer(MODEL_NAME)
collator = build_collator(tokenizer)

model_cfg = TeacherModelConfig(
    model_name = MODEL_NAME,
    gradient_checkpointing=True,
    hidden_dropout_prob = 0.1
)
model = build_teacher_model(model_cfg)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
LR_MAX = 2e-5 
DECAY_RATE = 0.95
WEIGHT_DECAY = 0.01

optimizer_grouped_parameters = get_llrd_optimizer_parameters(
    model, LR_MAX, WEIGHT_DECAY, DECAY_RATE
)
optimizer = torch.optim.AdamW(optimizer_grouped_parameters)

In [5]:
args = TrainingArguments(
    output_dir=str(paths.checkpoints / "bert_balanced_llrd"),
    per_device_train_batch_size=4,
    gradient_accumulation_steps=6, # 4*6 = 24 batch size
    num_train_epochs=4,            # Aumentato da 2 a 4
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,                # Valuta più spesso
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    warmup_ratio=0.1,
    learning_rate=LR_MAX,
    lr_scheduler_type="linear",
    bf16=bf16_supported(),
    fp16=not bf16_supported(),
    metric_for_best_model="pr_auc", # Ottimizziamo per Area Under PR Curve (Threshold independent)
    greater_is_better=True,
    report_to="none"
)

In [6]:
class CustomTrainer(Trainer):
    def __init__(self, *args, pos_weight_value=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Salviamo il valore del peso (es. 2.8)
        self.pos_weight_value = pos_weight_value

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. Rimuoviamo labels dall'input per evitare calcoli automatici errati
        labels = inputs.pop("labels")
        
        # 2. Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # 3. Fix Dimensioni [Batch, 1] -> [Batch]
        if logits.shape != labels.shape:
            logits = logits.view(-1)
            labels = labels.view(-1)
            
        # 4. Configurazione Loss con Peso
        if self.pos_weight_value is not None:
            # Creiamo il tensore peso sullo stesso device (GPU) dei logits
            weight_tensor = torch.tensor([self.pos_weight_value], device=logits.device)
            loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=weight_tensor)
        else:
            loss_fct = torch.nn.BCEWithLogitsLoss()
            
        loss = loss_fct(logits, labels.float())
        
        return (loss, outputs) if return_outputs else loss

In [7]:
trainer = CustomTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None)
)

print("🚀 Starting Training on Balanced Data (50/50)...")
trainer.train()

🚀 Starting Training on Balanced Data (50/50)...


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc,Pr Auc
200,0.665700,0.666018,0.620000,0.369794,0.686183,0.480590,0.688947,0.414698
400,0.624300,0.588046,0.698600,0.433215,0.572209,0.493105,0.718985,0.455853
600,0.598300,0.614102,0.662200,0.409735,0.722873,0.523016,0.741993,0.493376
800,0.580700,0.567127,0.693600,0.437593,0.686963,0.534629,0.758814,0.527168
1000,0.578800,0.583624,0.687400,0.435262,0.740047,0.548135,0.773156,0.548046
1200,0.577700,0.593276,0.697200,0.445586,0.744731,0.557569,0.782158,0.562258
1400,0.594200,0.559299,0.720600,0.468884,0.682279,0.555803,0.778289,0.564880
1600,0.573000,0.571092,0.710600,0.458207,0.710383,0.557086,0.779506,0.565830
1800,0.554000,0.547624,0.714000,0.462544,0.718189,0.562691,0.787689,0.583970
2000,0.594700,0.576559,0.689600,0.437298,0.737705,0.549099,0.783887,0.573833


KeyboardInterrupt: 